# steering-xling on Colab

Runs the sweep on a GPU runtime. By default it runs the **smoke grid on the main model** (`Qwen/Qwen2.5-1.5B-Instruct`). For the full sweep, set `GRID = "full"` and a new `RUN_NAME` in the next cell. The full sweep took about 5.6 hours on a T4.

**Before you start**
1. Runtime > Change runtime type > **T4 GPU**.
2. Optional: in the key icon (Secrets), add `HF_TOKEN` for faster Hugging Face downloads.

`results/` is linked to a folder in your Google Drive, so a disconnect loses nothing. Rerun the run cell and the sweep resumes from the rows it already wrote.

In [ ]:
# Settings for this run. GRID is "smoke" or "full"; MODEL is "main" (1.5B) or "dev" (0.5B).
BRANCH = "main"
GRID = "smoke"
MODEL = "main"
RUN_NAME = "t4-smoke-1.5b"   # folder name under MyDrive/steering-xling-runs/

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: switch the runtime to T4 GPU and run again."
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("No HF_TOKEN secret: downloads run unauthenticated (slower, still fine).")
if not os.path.isdir("/content/steering-xling"):
    !git clone -q --branch {BRANCH} https://github.com/Muhanad-husn/steering-xling.git /content/steering-xling
%cd /content/steering-xling
!git log --oneline -1

In [ ]:
# Pinned versions from requirements.txt. torch is reinstalled at the pinned version.
!pip install -q -r requirements.txt
# Colab's torchvision is built for its own torch and breaks `import transformers` once the pinned
# torch replaces it. The project does not use torchvision, so remove it.
!pip uninstall -y -q torchvision
!python -c "import torch, transformers; print('torch', torch.__version__, 'cuda', torch.version.cuda, '| transformers', transformers.__version__)"

In [ ]:
# Link results/ to Drive so the sweep can resume after a disconnect.
from google.colab import drive
drive.mount("/content/drive")
OUT = f"/content/drive/MyDrive/steering-xling-runs/{RUN_NAME}"
os.makedirs(OUT, exist_ok=True)
if os.path.isdir("results") and not os.path.islink("results"):
    !rm -rf results
if not os.path.islink("results"):
    os.symlink(OUT, "results")
print("results ->", os.path.realpath("results"))

In [ ]:
# The run. Safe to rerun: conditions already in results.csv are skipped.
!python -m pytest -q 2>&1 | tail -1
!/usr/bin/env bash -c "time python -m src.run_sweep --config config.yaml --grid {GRID} --model {MODEL}" 2>&1 | grep -v "Loading weights" | tee -a results/run.log

In [ ]:
# Summary, and a readable file of every Arabic output for reading.
import json, pandas as pd
df = pd.read_csv("results/results.csv", dtype={"fingerprint": str})
print("fingerprints:", df["fingerprint"].unique(), "| commits:", df["git_sha"].unique())
cols = ["layer", "strength", "lang", "vector_type", "sentiment_a_mean", "sentiment_b_mean", "perplexity", "arabic_share"]
display(df[df["vector_type"] != "split_half"][cols].round(3))
display(df[df["vector_type"] == "split_half"][["layer", "split_half_cos_mean", "split_half_cos_min"]].round(3))
rows = [json.loads(l) for l in open("results/samples.jsonl", encoding="utf-8")]
with open("results/arabic-outputs.md", "w", encoding="utf-8") as f:
    for r in rows:
        if r["lang"] == "ar":
            f.write(f"## {r['vector_type']} layer={r['layer']} strength={r['strength']} seed={r['seed']} {r['prompt_id']}\n\n")
            f.write(f"**message:** {r['user_message']}\n\n" + (f"**system:** {r['system']}\n\n" if r["system"] else ""))
            f.write("> " + r["text"].strip().replace("\n", "\n> ") + "\n\n")
print("Saved to Drive:", sorted(os.listdir(OUT)))

## After the run

Everything is in `MyDrive/steering-xling-runs/<RUN_NAME>/`:
- `results.csv`
- `samples.jsonl`
- `run.log`
- `arabic-outputs.md`

To keep a run record, copy `results.csv`, `samples.jsonl` and `run.log` into `logs/<date>-<RUN_NAME>/` with a short README, as the existing records do.

Read `arabic-outputs.md` before any Arabic claim is made.